In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('adi-dev')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/21 19:50:28 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/21 19:50:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/21 19:50:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/21 19:50:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
from pyspark.sql import functions as F

from adi.io import CsvStore, TrialBalanceRepository
from adi.pipeline import TrialBalancePipeline
from adi.config.settings import TABLE_PATHS
from adi.enrichments import (
    TransformationManager, 
    ReferenceManager
)

from finmap import FinMapClient


def display_df(df):
    display(df.toPandas())

In [3]:
store = CsvStore(spark, table_paths=TABLE_PATHS)
repository = TrialBalanceRepository(store)

transformation_manager = TransformationManager(spark)
reference_manager = ReferenceManager(spark)

finmap = FinMapClient.from_csv(
    spark=spark,
    metadata_path='data/reference/mapping_meta.csv',
    data_path='data/reference/mapping_data.csv',
)

pipeline = TrialBalancePipeline(
    transformation_manager=transformation_manager,
    reference_manager=reference_manager,
    finmap=finmap,
)

In [4]:
df_source = repository.read_source()

df_staging = pipeline.staging(df_source)
repository.write_staging(df_staging)

df_staging_reloaded = repository.read_staging()

display_df(df_staging_reloaded)

,BATCH_ID,EXTRACT_DT,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_APP_NM,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,...,ENTITY_SUN_ID,CLIENT_ID_TYPE,INTERGROUP_IND,POSTING_MEASURE_NM,MEASURE_TYPE,POSTING_MEASURE_FUNC_CCY_CD,POSTING_MEASURE_TRANS_AMT,FX_RATE,POSTING_MEASURE_FUNC_AMT,CR_DR_EVALUATOR
0,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-14,NSA,4031,100006,...,100774,THIRDPARTY,TP,PREVIOUS_DAY_BALANCE,REPORTABLE,USD,-1100000.000000000000,1.000000000000,-1100000.000000000000,DEBIT
1,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-14,NSA,4031,100006,...,100774,THIRDPARTY,TP,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT
2,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-14,NSA,4031,100006,...,100774,THIRDPARTY,TP,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT
3,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-14,NSA,4031,100006,...,100774,THIRDPARTY,TP,CURRENT_DAY_EOD_BALANCE,REPORTABLE,USD,-1100000.000000000000,1.000000000000,-1100000.000000000000,DEBIT
4,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-14,NSA,4031,100006,...,100774,THIRDPARTY,TP,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-11,NSA,4013,999999,...,100774,THIRDPARTY,TP,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT
92,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-11,NSA,4013,999999,...,100774,THIRDPARTY,TP,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT
93,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-11,NSA,4013,999999,...,100774,THIRDPARTY,TP,CURRENT_DAY_EOD_BALANCE,REPORTABLE,USD,499589614.620000000000,1.000000000000,499589614.620000000000,CREDIT
94,1,2025-03-31,2025-03-31,2025-03-31,11392,IMPACT,rec-11,NSA,4013,999999,...,100774,THIRDPARTY,TP,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,CREDIT


In [5]:
default_df = finmap.get_mapping('attr_reference_mapping').data

In [6]:
display_df(default_df.orderBy('src_app_cd', 'src_entity_cd', 'dataclass'))

,SRC_APP_CD,SRC_ENTITY_CD,DATACLASS,POSTING_RULE_ID,RULE_ID_DESC,ATTR_REFERENCE_NAME,ATTR_REFERENCE_VALUE,WEIGHTAGE
0,*,*,*,*,*,@SRC_ACCOUNT_ID,SRC_ACCOUNT_ID,00000100000000000000
1,*,*,TRIAL_BALANCE,*,*,@NORM_ACCT_SIGN,NORM_ACCT_SIGN,00100100000000000000


In [7]:
finmap._manager.get_attribute_references()

{'@NORM_ACCT_SIGN': 'NORM_ACCT_SIGN', '@SRC_ACCOUNT_ID': 'SRC_ACCOUNT_ID'}